In [ ]:
import os
import glob

import json
import numpy as np
import pandas as pd

from pathlib import Path
from typing import List, Tuple, Any, Optional
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor

from concurrent.futures import ProcessPoolExecutor, as_completed

In [2]:
# STORAGE_DIR = "Z:\\starcraft-vision\\"
STORAGE_DIR = "/mnt/nas/baecm/starcraft-vision/"

DATA_DIR = os.path.join(STORAGE_DIR, "data")
GROUND_TRUTH_DIR = os.path.join(DATA_DIR, "label", "dst")
PREDICTIONS_DIR = os.path.join(STORAGE_DIR, "predictions")

In [34]:
REPLAYS = [36, 212, 438, 522, 1660, 6254]
# REPLAYS = [1559, 1628, 2351, 6219, 11251]
# REPLAYS = [275, 1725, 3613, 4520, 4664]
# REPLAYS = [36, 212, 438, 522, 1660, 6254, 1559, 1628, 2351, 6219, 11251, 275, 1725, 3613, 4520, 4664]

REPLAYS = [str(_) + '.rep' for _ in REPLAYS]

In [ ]:
# TARGET_MODEL = "test_input_model/model_020"
# TARGET_MODEL = "all_correct_win4_b16_20250826_045543/model_030"
TARGET_MODEL = "all_correct_win4_b16_kbrs_20250821_044349/model_015"

In [36]:
ground_truth_json_path = [
    f for name in REPLAYS
    for f in glob.glob(os.path.join(GROUND_TRUTH_DIR, name, '*.json'))
]

In [37]:
def _build_json_paths(directory: str,
                      label_method: str = "all_correct",
                      replays: Optional[List[str]] = None) -> List[str]:
    directory = os.path.abspath(directory)
    if replays:
        return [os.path.join(directory, rep, f"{label_method}.json") for rep in replays]
    return [str(p) for p in Path(directory).glob("**/*.json")]

def _read_one_json(path: str, use_orjson: bool = True) -> Tuple[str, Any, Optional[str]]:
    try:
        if use_orjson:
            try:
                import orjson
                with open(path, "rb") as f:
                    return path, orjson.loads(f.read()), None
            except ModuleNotFoundError:
                pass  # orjson이 없으면 표준 json로 폴백
        with open(path, "r", encoding="utf-8") as f:
            return path, json.load(f), None
    except Exception as e:
        return path, None, f"{type(e).__name__}: {e}"

def read_json_files_threaded(directory: str,
                             label_method: str = "all_correct",
                             replays: Optional[List[str]] = None,
                             max_workers: Optional[int] = None) -> List[Any]:
    paths = _build_json_paths(directory, label_method, replays)
    if not paths:
        return []

    if max_workers is None:
        cpu = os.cpu_count() or 4
        max_workers = min(32, cpu * 5)

    results: List[Any] = []
    errors: List[str] = []

    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        for path, data, err in ex.map(_read_one_json, paths):
            if err:
                errors.append(f"[FAIL] {path} -> {err}")
            else:
                results.append(data)

    if errors:
        print("\n".join(errors))
    return results

def read_json_files_processes(directory: str,
                              label_method: str = "all_correct",
                              replays: Optional[List[str]] = None,
                              max_workers: Optional[int] = None) -> List[Any]:
    paths = _build_json_paths(directory, label_method, replays)
    if not paths:
        return []

    if max_workers is None:
        max_workers = os.cpu_count() or 4

    results: List[Any] = []
    errors: List[str] = []

    with ProcessPoolExecutor(max_workers=max_workers) as ex:
        for path, data, err in ex.map(_read_one_json, paths):
            if err:
                errors.append(f"[FAIL] {path} -> {err}")
            else:
                results.append(data)

    if errors:
        print("\n".join(errors))
    return results

In [38]:
ground_truth_data = read_json_files_processes(GROUND_TRUTH_DIR, label_method='all_correct', replays=REPLAYS, max_workers=8)

In [39]:
prediction_data = read_json_files_processes(os.path.join(PREDICTIONS_DIR, TARGET_MODEL), label_method='all_correct', replays=REPLAYS, max_workers=8)

[FAIL] /mnt/nas/baecm/starcraft-vision/predictions/all_correct_win4_b16_kbrs_20250821_044349/model_014/6254.rep/all_correct.json -> FileNotFoundError: [Errno 2] No such file or directory: '/mnt/nas/baecm/starcraft-vision/predictions/all_correct_win4_b16_kbrs_20250821_044349/model_014/6254.rep/all_correct.json'


In [40]:
def get_annotations(data, replay_suffix=".rep"):
    frames = []

    for item in (data or []):
        anns = item.get("annotations", [])
        if not anns:
            continue

        df = pd.DataFrame(anns)

        desc = (item.get("info") or {}).get("description", "") or ""
        replay_id = None
        if isinstance(desc, str) and desc.strip():
            replay_id = desc.strip().split()[-1]

        if replay_id is None:
            replay_id = "unknown"

        if replay_suffix and not str(replay_id).endswith(replay_suffix):
            replay_id = f"{replay_id}{replay_suffix}"

        df["replay"] = replay_id
        frames.append(df)

    base_cols = ["replay", "id", "image_id", "category_id", "bbox", "area", "segmentation", "iscrowd"]
    if not frames:
        return pd.DataFrame(columns=base_cols + ["score"])

    has_score = any("score" in f.columns for f in frames)

    wanted_cols = base_cols + (["score"] if has_score else [])
    for i, f in enumerate(frames):
        for col in wanted_cols:
            if col not in f.columns:
                frames[i][col] = pd.NA

    annotations = pd.concat(frames, ignore_index=True)

    return annotations[wanted_cols]

In [41]:
annotation_ground_truth = get_annotations(ground_truth_data)
annotation_prediction = get_annotations(prediction_data)

In [42]:
def dataframe_topk_score(dataframe, K=1):
    topk_df = (
      dataframe.sort_values('score', ascending=False)
        .groupby(['replay', 'image_id'], group_keys=False)
        .head(K)
    )
    return topk_df.sort_values(['replay', 'image_id'])

In [43]:
top1_annotation_prediction = dataframe_topk_score(annotation_prediction, K=1)

In [44]:
annotation_ground_truth["idx"] = annotation_ground_truth.groupby(["replay", "image_id"]).cumcount()  # 0~4 인덱스 생성
wide_ground_truth = annotation_ground_truth.pivot(index=["replay", "image_id"], columns="idx", values="bbox")
wide_ground_truth.columns = [f"gt_{i}" for i in wide_ground_truth.columns]
wide_ground_truth.reset_index(inplace=True)


In [45]:
top1_annotation_prediction["idx"] = top1_annotation_prediction.groupby(["replay", "image_id"]).cumcount()  # 0~4 인덱스 생성
wide_prediction = top1_annotation_prediction.pivot(index=["replay", "image_id"], columns="idx", values="bbox")
wide_prediction.columns = [f"pred_{i}" for i in wide_prediction.columns]
wide_prediction.reset_index(inplace=True)

In [46]:
result_inference = pd.merge(wide_ground_truth, wide_prediction,on=["replay", "image_id"], how="outer")
result_inference = result_inference.bfill()
result_inference = result_inference.dropna()

new_cols = []

for c in result_inference.columns:
    if c in ('replay', 'image_id'):
        new_cols.append(('meta', c))
    elif c.startswith('gt_'):
        new_cols.append(('gt', c.split('_')[1]))
    elif c.startswith('pred_'):
        new_cols.append(('pred', c.split('_')[1]))
    else:
        new_cols.append(('other', c))  # fallback

result_inference.columns = pd.MultiIndex.from_tuples(new_cols)

In [47]:
result_inference.head()

meta                         gt                                      \
     replay image_id                 0                 1                 2   
0  1660.rep        0  [108, 2, 20, 12]  [108, 2, 20, 12]  [108, 2, 20, 12]   
1  1660.rep        1  [108, 2, 20, 12]  [108, 2, 20, 12]  [108, 2, 20, 12]   
2  1660.rep        2  [108, 2, 20, 12]  [108, 2, 20, 12]  [108, 2, 20, 12]   
3  1660.rep        3  [108, 1, 20, 12]  [108, 2, 20, 12]  [108, 2, 20, 12]   
4  1660.rep        4  [108, 1, 20, 12]  [108, 2, 20, 12]  [108, 2, 20, 12]   

                                                 pred  
                  3                 4               0  
0  [108, 2, 20, 12]  [108, 2, 20, 12]  [0, 1, 19, 12]  
1  [108, 2, 20, 12]  [108, 2, 20, 12]  [0, 1, 19, 12]  
2  [108, 2, 20, 12]  [108, 2, 20, 12]  [0, 1, 19, 12]  
3  [108, 2, 20, 12]  [108, 2, 20, 12]  [0, 1, 19, 12]  
4  [108, 2, 20, 12]  [108, 2, 20, 12]  [0, 1, 19, 12]

In [48]:
def union_metrics_singlepass(rows, gt_cols, pred_cols,
                             grid_w, grid_h, orig_w, orig_h,
                             kernel_x, kernel_y, use_bbox_size, thresholds,
                             return_summary: bool = False):
    n = len(rows)
    ic_multi = np.full(n, np.nan, dtype=float)
    ic_ratio = np.zeros(n, dtype=float)
    ic_flags = {t: np.zeros(n, dtype=float) for t in thresholds}

    def xywh_to_rect(x,y,w,h):
        if not use_bbox_size:
            w,h = kernel_x, kernel_y
        rx = int(round(x * 32 / orig_w * (grid_w - w)))
        ry = int(round(y * 32 / orig_h * (grid_h - h)))
        rx = max(0, min(grid_w - w, rx))
        ry = max(0, min(grid_h - h, ry))
        return rx, ry, w, h

    for j, (_, row) in enumerate(rows.iterrows()):
        total = np.zeros((grid_h, grid_w), dtype=np.int16)
        have_gt = False
        for gc in gt_cols:
            if gc in row and row[gc] is not None and not (isinstance(row[gc], float) and np.isnan(row[gc])):
                x,y,w,h = (row[gc] if len(row[gc])==4 else tuple(row[gc])[:4])
                rx,ry,rw,rh = xywh_to_rect(x,y,w,h)
                total[ry:ry+rh, rx:rx+rw] += 1
                have_gt = True

        pred_mask = np.zeros((grid_h, grid_w), dtype=bool)
        for pc in pred_cols:
            if pc in row and row[pc] is not None and not (isinstance(row[pc], float) and np.isnan(row[pc])):
                x,y,w,h = (row[pc] if len(row[pc])==4 else tuple(row[pc])[:4])
                rx,ry,rw,rh = xywh_to_rect(x,y,w,h)
                pred_mask[ry:ry+rh, rx:rx+rw] = True

        if not have_gt or not pred_mask.any():
            ic_multi[j] = np.nan
            ic_ratio[j] = 0.0
            for t in thresholds:
                ic_flags[t][j] = 0.0
            continue

        vals = total[pred_mask]
        ic_multi[j] = vals.mean()
        ic_ratio[j] = (vals > 0).mean()
        for t in thresholds:
            ok = (ic_ratio[j] > t) if t == 0.0 else (ic_ratio[j] >= t)
            ic_flags[t][j] = float(ok)

    out = pd.DataFrame({
        ("union","ic_multi"): ic_multi,
        ("union","ic_ratio"): ic_ratio,
        **{("union", f"ic@{int(round(t*100)):03d}"): ic_flags[t] for t in thresholds}
    }, index=rows.index)

    return out

In [49]:
def _union_metrics_worker(args):
    (sub_df, gt_cols, pred_cols, grid_w, grid_h, orig_w, orig_h,
     kernel_x, kernel_y, use_bbox_size, thresholds) = args
    return union_metrics_singlepass(sub_df, gt_cols, pred_cols, grid_w, grid_h,
                                    orig_w, orig_h, kernel_x, kernel_y,
                                    use_bbox_size, thresholds)

In [50]:
def add_union_metrics_parallel(
    df,
    gt_cols=(("gt","0"),("gt","1"),("gt","2"),("gt","3"),("gt","4")),
    pred_cols=(("pred","0"),),
    grid_w=128, grid_h=128,
    orig_w=3456, orig_h=3720,
    kernel_x=20, kernel_y=12,
    use_bbox_size=False,
    thresholds=(0.0, 0.3, 0.5),
    max_workers=None, chunksize=None
):
    out_cols = [("union","ic_multi"), ("union","ic_ratio")] + \
               [("union", f"ic@{int(round(t*100)):03d}") for t in thresholds]
    for c in out_cols:
        if c not in df.columns:
            df[c] = np.nan

    idx = df.index.to_numpy()
    if chunksize is None:
        nworkers = max_workers or os.cpu_count() or 4
        chunksize = max(1000, len(idx)//(nworkers*4) or 1)
    splits = [idx[i:i+chunksize] for i in range(0, len(idx), chunksize)]

    arglist = [(df.loc[s].copy(), gt_cols, pred_cols, grid_w, grid_h,
                orig_w, orig_h, kernel_x, kernel_y, use_bbox_size, thresholds)
               for s in splits]

    results = []
    with ProcessPoolExecutor(max_workers=max_workers) as ex:
        futs = [ex.submit(_union_metrics_worker, a) for a in arglist]
        for f in as_completed(futs):
            results.append(f.result())

    out = pd.concat(results).sort_index()
    df.loc[out.index, out.columns] = out.values
    return df

In [51]:
out = add_union_metrics_parallel(
    result_inference,
    gt_cols = (("gt", "0"), ("gt", "1"), ("gt", "2"), ("gt", "3"), ("gt", "4")),
    pred_cols = (("pred", "0"),),
    use_bbox_size=False,
    thresholds=(0.0, 0.3, 0.5)
)

out

meta                          gt                     \
          replay image_id                  0                  1   
0       1660.rep        0   [108, 2, 20, 12]   [108, 2, 20, 12]   
1       1660.rep        1   [108, 2, 20, 12]   [108, 2, 20, 12]   
2       1660.rep        2   [108, 2, 20, 12]   [108, 2, 20, 12]   
3       1660.rep        3   [108, 1, 20, 12]   [108, 2, 20, 12]   
4       1660.rep        4   [108, 1, 20, 12]   [108, 2, 20, 12]   
...          ...      ...                ...                ...   
140573   522.rep    27879  [31, 107, 20, 12]  [28, 105, 20, 12]   
140574   522.rep    27880  [31, 107, 20, 12]  [28, 105, 20, 12]   
140575   522.rep    27881  [31, 107, 20, 12]  [28, 105, 20, 12]   
140576   522.rep    27882  [31, 107, 20, 12]  [28, 105, 20, 12]   
140577   522.rep    27883  [31, 107, 20, 12]  [28, 105, 20, 12]   

                                                                 \
                        2                  3                  4   
0        [108, 2, 20, 12]   [108, 2, 20, 12]   [108, 2, 20, 12]   
1        [108, 2, 20, 12]   [108, 2, 20, 12]   [108, 2, 20, 12]   
2        [108, 2, 20, 12]   [108, 2, 20, 12]   [108, 2, 20, 12]   
3        [108, 2, 20, 12]   [108, 2, 20, 12]   [108, 2, 20, 12]   
4        [108, 2, 20, 12]   [108, 2, 20, 12]   [108, 2, 20, 12]   
...                   ...                ...                ...   
140573  [29, 107, 20, 12]  [29, 104, 20, 12]  [31, 108, 20, 12]   
140574  [29, 107, 20, 12]  [29, 104, 20, 12]  [31, 108, 20, 12]   
140575  [29, 107, 20, 12]  [29, 104, 20, 12]  [31, 108, 20, 12]   
140576  [29, 107, 20, 12]  [29, 104, 20, 12]  [31, 108, 20, 12]   
140577  [29, 107, 20, 12]  [29, 104, 20, 12]  [31, 108, 20, 12]   

                     pred     union                                
                        0  ic_multi ic_ratio ic@000 ic@030 ic@050  
0          [0, 1, 19, 12]  0.000000      0.0    0.0    0.0    0.0  
1          [0, 1, 19, 12]  0.000000      0.0    0.0    0.0    0.0  
2          [0, 1, 19, 12]  0.000000      0.0    0.0    0.0    0.0  
3          [0, 1, 19, 12]  0.000000      0.0    0.0    0.0    0.0  
4          [0, 1, 19, 12]  0.000000      0.0    0.0    0.0    0.0  
...                   ...       ...      ...    ...    ...    ...  
140573  [29, 106, 20, 12]  4.195833      1.0    1.0    1.0    1.0  
140574  [29, 106, 20, 12]  4.195833      1.0    1.0    1.0    1.0  
140575  [29, 106, 20, 12]  4.195833      1.0    1.0    1.0    1.0  
140576  [29, 106, 20, 12]  4.195833      1.0    1.0    1.0    1.0  
140577  [29, 106, 20, 12]  4.195833      1.0    1.0    1.0    1.0  

[140578 rows x 13 columns]

In [52]:
gt_cols = result_inference.columns[
    out.columns.get_level_values(0).str.startswith("union")
]

g = out.groupby(("meta", "replay"))[gt_cols].mean()

out_per_replay = g.T.groupby(level=1).mean().T

out_per_replay

,ic@000,ic@030,ic@050,ic_multi,ic_ratio
"(meta, replay)",,,,,
1660.rep,0.832555,0.802197,0.768203,1.535563,0.706127
212.rep,0.801218,0.771060,0.732833,1.312016,0.664069
36.rep,0.905154,0.884045,0.858066,1.919485,0.796694
438.rep,0.904401,0.892731,0.871757,1.913800,0.805583
522.rep,0.884127,0.864976,0.847296,1.894284,0.788756


In [53]:
out_per_replay.describe()

,ic@000,ic@030,ic@050,ic_multi,ic_ratio
count,5.000000,5.000000,5.000000,5.000000,5.000000
mean,0.865491,0.843002,0.815631,1.715030,0.752246
std,0.046503,0.053575,0.061356,0.277521,0.063355
min,0.801218,0.771060,0.732833,1.312016,0.664069
25%,0.832555,0.802197,0.768203,1.535563,0.706127
50%,0.884127,0.864976,0.847296,1.894284,0.788756
75%,0.904401,0.884045,0.858066,1.913800,0.796694
max,0.905154,0.892731,0.871757,1.919485,0.805583


In [31]:
out_per_replay.loc[['36.rep', '212.rep', '438.rep', '522.rep', '1660.rep']].describe()

,ic@000,ic@030,ic@050,ic_multi,ic_ratio
count,5.000000,5.000000,5.000000,5.000000,5.000000
mean,0.770233,0.734158,0.695057,1.414164,0.628084
std,0.062065,0.066113,0.069660,0.232467,0.063701
min,0.685686,0.642751,0.596630,1.071051,0.535567
25%,0.722494,0.685246,0.646084,1.275958,0.586821
50%,0.806054,0.771016,0.738201,1.549308,0.671804
75%,0.813705,0.782294,0.743452,1.563231,0.672036
max,0.823226,0.789481,0.750917,1.611273,0.674192


In [ ]:
def compute_total_intersection_percent(out: pd.DataFrame):
    if ("meta", "replay") not in out.columns:
        if out.index.names and "meta" in out.index.names and "replay" in out.index.names:
            out = out.reset_index()
        else:
            raise KeyError("('meta','replay')가 컬럼/인덱스 어디에도 없습니다.")

    if ("union", "ic_ratio") not in out.columns:
        raise KeyError("('union','ic_ratio') 컬럼이 없습니다. add_union_metrics_parallel 결과를 확인하세요.")

    overall_total = float(out[("union", "ic_ratio")].mean())

    g = out.groupby(("meta", "replay"))[[("union", "ic_ratio")]].mean()
    out_per_replay = g.T.groupby(level=1).mean().T
    out_per_replay = out_per_replay.rename(columns={"ic_ratio": "total_intersection_percent"})

    return overall_total, out_per_replay

In [54]:
# compute_total_intersection_percent(out)